In [ ]:
!git clone https://github.com/MMMU-Benchmark/MMMU

In [ ]:
!pip install datasets transformers torch Pillow tqdm

In [ ]:
%cd MMMU
%cd mmmu-pro

In [ ]:
!pip install -r requirements.txt

In [ ]:
import torch
import json
import re
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoModelForVision2Seq, AutoProcessor
from PIL import Image


# ──────────────────────────────────────────
# 1. LOAD MODEL
# ──────────────────────────────────────────
model_id = "ibm-granite/granite-vision-3.3-2b"

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForVision2Seq.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
model.eval()
print(f"✅ Model loaded: {model_id}")

# ──────────────────────────────────────────
# 2. LOAD MMMU-PRO DATASET
# ──────────────────────────────────────────
# Subsets: "standard" or "vision"
dataset = load_dataset("MMMU/MMMU_Pro", 'standard (4 options)', split="test")
print(f"✅ Dataset loaded: {len(dataset)} samples")

# ──────────────────────────────────────────
# 3. PROMPT FORMATTERS
# ──────────────────────────────────────────
def build_prompt(sample, num_images):
    question    = sample["question"]
    options     = sample["options"]
    options_str = "\n".join(options) if isinstance(options, list) else options

    # One <image> token per image — must match number of images passed to processor
    image_tokens = "\n".join(["<image>"] * num_images)

    prompt = (
        f"{image_tokens}\n\n"
        f"Look at the image(s) carefully and answer the following question.\n\n"
        f"Question: {question}\n\n"
        f"Options:\n{options_str}\n\n"
        f"Answer with only the option letter (A, B, C, or D)."
    )
    return prompt


def build_prompt_text_only(sample):
    question    = sample["question"]
    options     = sample["options"]
    options_str = "\n".join(options) if isinstance(options, list) else options

    prompt = (
        f"Answer the following question.\n\n"
        f"Question: {question}\n\n"
        f"Options:\n{options_str}\n\n"
        f"Answer with only the option letter (A, B, C, or D)."
    )
    return prompt

# ──────────────────────────────────────────
# 4. IMAGE EXTRACTOR
# ──────────────────────────────────────────
def get_images(sample):
    images = []
    for key in ["image_1", "image_2", "image_3", "image_4", "image_5", "image_6", "image_7"]:
        img = sample.get(key)
        if img is not None:
            # Convert anything → PIL RGB
            if not isinstance(img, Image.Image):
                img = Image.fromarray(img)
            img = img.convert("RGB")
            images.append(img)
    return images

# ──────────────────────────────────────────
# 5. ANSWER EXTRACTOR
# ──────────────────────────────────────────
def extract_answer(raw_text):
    # Match first standalone A/B/C/D letter
    match = re.search(r'\b([A-D])\b', raw_text.strip())
    if match:
        return match.group(1)
    # Fallback: grab first character if it's a valid letter
    first = raw_text.strip()[:1].upper()
    return first if first in "ABCD" else "X"

# ──────────────────────────────────────────
# 6. RUN INFERENCE
# ──────────────────────────────────────────
results = []
correct = 0
total   = 0

for idx, sample in enumerate(tqdm(dataset, desc="Running inference")):
    try:
        images     = get_images(sample)
        num_images = len(images)
        gt         = sample["answer"].strip().upper()

        # Build prompt and inputs based on whether images are present
        if num_images > 0:
            prompt = build_prompt(sample, num_images)
            inputs = processor(
                text=prompt,
                images=images,
                return_tensors="pt"
            ).to(model.device)
        else:
            prompt = build_prompt_text_only(sample)
            inputs = processor(
                text=prompt,
                return_tensors="pt"
            ).to(model.device)

        # Generate
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=16,
                do_sample=False,
                temperature=None,
                top_p=None,
            )

        # Decode only the newly generated tokens (strip echoed prompt)
        input_len  = inputs["input_ids"].shape[1]
        new_tokens = output_ids[0][input_len:]
        raw_answer = processor.decode(new_tokens, skip_special_tokens=True)

        pred       = extract_answer(raw_answer)
        is_correct = (pred == gt)
        if is_correct:
            correct += 1
        total += 1

        results.append({
            "id":       sample.get("id", idx),
            "subject":  sample.get("subject", "unknown"),
            "question": sample["question"],
            "gt":       gt,
            "raw":      raw_answer,
            "pred":     pred,
            "correct":  is_correct,
        })

        # Live accuracy every 50 samples
        if (idx + 1) % 50 == 0:
            print(f"  [{idx+1}/{len(dataset)}] Running accuracy: {correct/total*100:.2f}%")

    except Exception as e:
        print(f"  ⚠️  Sample {idx} failed: {e}")
        results.append({
            "id":    sample.get("id", idx),
            "error": str(e),
        })

# ──────────────────────────────────────────
# 7. SAVE PREDICTIONS
# ──────────────────────────────────────────
output_file = "mmmu_pro_predictions.json"
with open(output_file, "w") as f:
    json.dump(results, f, indent=2)
print(f"\n✅ Predictions saved to {output_file}")

# ──────────────────────────────────────────
# 8. PER-SUBJECT BREAKDOWN
# ──────────────────────────────────────────
subject_scores = {}
for r in results:
    if "error" in r:
        continue
    subj = r.get("subject", "unknown")
    if subj not in subject_scores:
        subject_scores[subj] = {"correct": 0, "total": 0}
    subject_scores[subj]["total"]   += 1
    subject_scores[subj]["correct"] += int(r["correct"])

print("\n📊 Per-Subject Accuracy:")
print(f"  {'Subject':<35} {'Correct':>7} {'Total':>6} {'Acc':>7}")
print("  " + "-" * 58)
for subj, s in sorted(subject_scores.items()):
    acc = s["correct"] / s["total"] * 100
    print(f"  {subj:<35} {s['correct']:>7} {s['total']:>6} {acc:>6.1f}%")

# ──────────────────────────────────────────
# 9. FINAL SCORE
# ──────────────────────────────────────────
errors   = len([r for r in results if "error" in r])
accuracy = correct / total * 100 if total > 0 else 0

print("\n" + "=" * 45)
print(f"  MODEL   : {model_id}")
print(f"  SUBSET  : standard")
print(f"  TOTAL   : {total}")
print(f"  CORRECT : {correct}")
print(f"  ERRORS  : {errors}")
print(f"  ACCURACY: {accuracy:.2f}%")
print("=" * 45)

In [ ]:
!cat mmmu_pro_predictions.json
